In [1]:
# !pip -q install  fusion_solar_py pvlib retry-requests openmeteo_requests requests-cache python-telegram-bot

In [2]:
# !pip -q install git+https://github.com/EmaR97/EnergyManagementRL.git@testing-10

In [3]:
import os

from dotenv import load_dotenv

load_dotenv()


def get_env(name):
    return os.environ.get(name)

In [4]:
# from kaggle_secrets import UserSecretsClient
# 
# user_secrets = UserSecretsClient()
# 
# def get_env(name):
#     return user_secrets.get_secret(name)

In [5]:
FUSION_SOLAR_CLIENT_PASSWORD = get_env("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = get_env("FUSION_SOLAR_CLIENT_USERNAME")
LAT = float(get_env("LAT"))
LON = float(get_env("LON"))
TOKEN = get_env("TELEGRAM_BOT_TOKEN_TEST")
# TOKEN = get_env("TELEGRAM_BOT_TOKEN")
ADMIN_ID = int(get_env("TELEGRAM_ID"))

In [6]:
from energymanagementrl.utility import get_logger

logger_main = get_logger("Main")
logger_esm = get_logger("ESM")
logger_tb = get_logger("TB")

In [7]:
from energymanagementrl.production_forecast import *

panel_model = PanelModel(pdc0=0.42, temp_model_a=-3.56, temp_model_b=-0.075, delta_t=3, gamma_pdc=-0.004)
num_panels = 14
arrays = [ArrayConfig(name='sud_east', panel_model=panel_model, num_panels=num_panels, tilt_angle=25, azimuth=110),
          ArrayConfig(name='nord_west', panel_model=panel_model, num_panels=num_panels, tilt_angle=18, azimuth=290)]

_plant_config = PlantConfig(
    latitude=LAT, longitude=LON, timezone='Europe/Rome', inverter_pdc0=6, arrays=arrays
)
_production_forecaster = EnergyPredictionSystem(plant_config=_plant_config, open_meteo_client=OpenMeteoClient())

In [8]:
from energymanagementrl.fusion_solar_connector import *

_client = FusionSolarClientParsed(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                  huawei_subdomain="uni004eu5")
periodic_task = PeriodicTask(_client.keep_alive)
periodic_task.start()
_plant_id = _client.get_plant_ids()[0]
battery_id = _client.get_battery_ids(_plant_id)[0]

Periodic task started.


In [9]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from energymanagementrl.rl import load_model_with_weights

# Create a simple dummy environment used only to define the model struct, as the model will be used in inference
env = gym.Env()
env.action_space = spaces.Discrete(2)  # Two possible actions: 0 or 1
env.observation_space = spaces.Box(low=0, high=1000, shape=(55,), dtype=np.float64)  # 55 state variables

# Load the DQN policy
best_model = '../data/trained_models/models/dqn_1.0_0.06_0.06_0.02_1000_l_2.policy_weights.pth'
_model = load_model_with_weights(
    env,
    best_model,
    # 'dqn.2024_12_31_11_44_29.latest'
)

In [10]:
from energymanagementrl.rl.real_system_interaction import EnergyManagementSystem

system = EnergyManagementSystem(
    client=_client,
    plant_id=_plant_id,
    battery_id=battery_id,
    production_forecaster=_production_forecaster,
    model=_model,
    logger=logger_esm,
)

# system.set_active(True)

In [11]:
from energymanagementrl.interface import TelegramBot

bot = TelegramBot(
    system=system,
    token=TOKEN,
    allowed_users=[ADMIN_ID],
    logger=logger_tb,
)

2025-02-19 10:06:17,862 - TB - INFO - Handlers have been set up.
2025-02-19 10:06:17,863 - TB - INFO - TelegramBot initialized.


In [12]:
import asyncio
import nest_asyncio

nest_asyncio.apply()
loop = asyncio.get_event_loop()
loop.run_until_complete(bot.set_bot_commands())

2025-02-19 10:06:17,885 - TB - INFO - Setting bot commands...
2025-02-19 10:06:18,158 - TB - INFO - Bot commands set!


In [13]:
import threading
threading.Thread(target=lambda: asyncio.run(system.control_loop())).start()

2025-02-19 10:06:18,176 - ESM - INFO - Starting Control loop


In [14]:
from datetime import datetime


async def shutdown():
    await sleep_async()
    stopping_all()


def stopping_all():
    logger_main.warning("Stopping all...")
    system.stop_control_loop()
    bot.app.stop_running()


async def sleep_async(hour: int = None, minute: int = None):
    hour = hour or 11 if datetime.now().hour < 12 else 23
    minute = minute or 55
    now = datetime.now()
    stop_time = now.replace(hour=hour, minute=minute, second=0, microsecond=0)
    seconds = (stop_time - now).total_seconds()
    logger_main.info(f"Scheduled sleep until {stop_time}, sleeping for {seconds} seconds")
    await asyncio.sleep(seconds)
    logger_main.info(f"Scheduled sleep completed")


loop.create_task(
    shutdown()
    # sleep_async(7,51)
)

<Task pending name='Task-5' coro=<shutdown() running at /tmp/ipykernel_17398/1680221435.py:4>>

In [ ]:
loop.run_until_complete(bot.run())

2025-02-19 10:06:18,356 - TB - INFO - Bot is starting...
2025-02-19 10:06:19,535 - TB - INFO - Authorization check for user 5541452419.
2025-02-19 10:06:19,975 - TB - INFO - User 5541452419 requested system stats: {'battery_mode': 'FULLY_FEED_TO_GRID', 'soc': '43.0%', 'to_battery': '1.059kw', 'from_battery': '0kw', 'prod': '1.477kw', 'load': '0.418kw', 'to_grid': '0kw', 'from_grid': '0kw'}.
